In [78]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os

# ======================
# LOAD DATA
# ======================
df = pd.read_csv(r"C:\Users\kuash\Downloads\archive\Gold Price.csv")

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# ======================
# FEATURE ENGINEERING
# ======================
df['Lag1'] = df['Price'].shift(1)
df['Price_Diff'] = df['Price'] - df['Lag1']
df = df.dropna().reset_index(drop=True)

# ======================
# TRAIN TEST SPLIT
# ======================
split = int(len(df) * 0.8)

train_df = df.iloc[:split]
test_df = df.iloc[split:]

X_train = train_df[['Lag1']]
y_train = train_df['Price_Diff']

X_test = test_df[['Lag1']]
y_test = test_df['Price_Diff']

actual_price = test_df['Price'].values
lag_test = test_df['Lag1'].values

# ======================
# GRADIENT BOOSTING
# ======================
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train, y_train)

gb_pred = gb.predict(X_test)
gb_final = lag_test + gb_pred

# ======================
# XGBOOST
# ======================
xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42
)

xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)
xgb_final = lag_test + xgb_pred

# ======================
# EVALUATION FUNCTION
# ======================
def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n=== {name} MODEL RESULTS ===")
    print(f"MAE: {mae:.6f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"R2: {r2:.6f}")
    print(f"MAPE: {mape:.6f}")

# ======================
# PRINT RESULTS
# ======================
evaluate("GB", actual_price, gb_final)
evaluate("XGB", actual_price, xgb_final)

# ======================
# SAVE MODELS
# ======================
os.makedirs("models", exist_ok=True)

joblib.dump(gb, "models/gradient_boosting.pkl")
joblib.dump(xgb, "models/xgboost.pkl")


=== GB MODEL RESULTS ===
MAE: 948.164455
RMSE: 1195.599535
R2: 0.996632
MAPE: 1.125514

=== XGB MODEL RESULTS ===
MAE: 597.442307
RMSE: 897.808825
R2: 0.998101
MAPE: 0.687401


['models/xgboost.pkl']